Assignment || 


In [157]:
# data and toolkit 
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd


#datasets from scikit-learn 
from sklearn.datasets import load_diabetes
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris

## for the off-shell / scikit-learn models
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split



In [158]:
diabetes= load_diabetes(scaled=False)  #raw features unscaled
#now extract the features and target 
X=diabetes.data
Y=diabetes.target
# now will be using vectorization do check the shape always
print("featuer shape (X):",X.shape)
print("target shape (X):",Y.shape)

featuer shape (X): (442, 10)
target shape (X): (442,)


In [159]:
#Split protocol (60 / 20 / 20)
X_tr_val, X_te, y_tr_val, y_te = train_test_split(X, Y, test_size=0.20, random_state=42)

X_tr, X_va, y_tr, y_va = train_test_split(X_tr_val, y_tr_val, test_size=0.25, random_state=42)

# Print the sizes to verify they match assignment's sanity table
print(f"Train size: {X_tr.shape[0]}")  
print(f"Validation size: {X_va.shape[0]}")  
print(f"Test size: {X_te.shape[0]}")  
print(f"Total size: {X.shape[0]}")

Train size: 264
Validation size: 89
Test size: 89
Total size: 442


+general idea+
Standardizer in machine learning is the process of rescaling features so that they all share a mean of 0 and a standered deviation of 1 
so for every feature 
z= x-mean / deviation 
so each feature contributes equally to the gradient allowing the learning rate to work properly without blowing up 

In [160]:
#now the standlizer 
class Standardizer:
    def fit(self,X):
        #compute the mean and variance alog the axis 0 --- columns
        self.mu=np.mean(X,axis=0)
        self.sigma=np.std(X,axis=0)
        #handle the assignment restriction on std 0-----> 1 when zero variance column
        self.sigma[self.sigma==0]=1.0
    def transform(self, X):
        return (X-self.mu)/self.sigma
    def fit_transform(self,X):
        self.fit(X)
        return self.fit_transform(X)

scaler =Standardizer()
scaler.fit(X_tr)  # Fit ONLY on training data!
X_tr_scaled = scaler.transform(X_tr)
X_va_scaled = scaler.transform(X_va)
X_te_scaled = scaler.transform(X_te)
def add_bias(X):
# Adds a column of ones to the left of X -> shape becomes (N, D+1)
    return np.hstack([np.ones((X.shape[0], 1)), X])

# Ready-to-use matrices with bias
X_tr_tilde = add_bias(X_tr_scaled)

why must μ and σ come from the training rows only, and how would computing them on all rows
make your test result misleading?
-there will be data leakage is we compute m , std from all the dataset so the model takes unfair peek at the test distribuation and that will lead to a misleading model 


In [161]:
#regression matrics 
def mse(y_true,y_pred):
    return np.mean((y_true-y_pred)**2)

def rmse(y_true,y_pred):
    return np.sqrt(mse(y_true,y_pred))

def r2(y_true,y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

In [162]:
#classification - cunfusion matrics
def accuracy(y_true,y_pred):
    return np.mean(y_true==y_pred)

def confusion_matrix(y_true,y_pred,K):   # k number of classes / how many categories we're trying to predict
    cm=np.zeros((K,K),dtype=int)
    for t , p in zip(y_true,y_pred):      # zip takes t , p and pair them together as i ++
        cm[t,p]+=1
    return cm

def classification_matrics(cm):
    k=cm.shape[0]
    precisions = []
    recalls=[]
    f1s=[]

    for i in range(K):
        tp=cm[i,i]
        fp=np.sum(cm[:,i])-tp # Column sum minus true positives
        fn=np.sum(cm[i,:])-tp # Row sum minus true positives

        precision = tp/(tp+fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (
            (2 * precision * recall) / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
        
  # Macro averages
    return (
        np.mean(precisions),
        np.mean(recalls),
        np.mean(f1s),
        precisions,
        recalls,
        f1s,
    )

        



In [163]:
#used Ai to test 
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix as sk_confusion_matrix
from sklearn.metrics import precision_recall_fscore_support

# 1. Generate random labels for K = 4 classes and N = 500 samples
rng = np.random.default_rng(42)
K = 4
N = 500
y_true = rng.integers(0, K, size=N)
y_pred = rng.integers(0, K, size=N)

#-- VERIFY CONFUSION MATRIX ---
my_cm = confusion_matrix(y_true, y_pred, K)
sk_cm = sk_confusion_matrix(y_true, y_pred, labels=np.arange(K))

assert np.allclose(
    my_cm, sk_cm, atol=1e-12
), "Confusion matrix does not match sklearn!"
print("✅ Confusion Matrix matches sklearn to 1e-12")

# --- VERIFY ACCURACY ---
my_acc = accuracy(y_true, y_pred)
sk_acc = accuracy_score(y_true, y_pred)

assert np.isclose(
    my_acc, sk_acc, atol=1e-12
), "Accuracy does not match sklearn!"
print("✅ Accuracy matches sklearn to 1e-12")

# --- VERIFY PRECISION, RECALL, F1 (MACRO) ---
# Note: sklearn's precision_recall_fscore_support handles zero division via zero_division=0
my_macro_p, my_macro_r, my_macro_f1, _, _, _ = classification_matrics(my_cm)
sk_macro_p, sk_macro_r, sk_macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)

assert np.isclose(
    my_macro_p, sk_macro_p, atol=1e-12
), "Macro Precision mismatch!"
assert np.isclose(my_macro_r, sk_macro_r, atol=1e-12), "Macro Recall mismatch!"
assert np.isclose(my_macro_f1, sk_macro_f1, atol=1e-12), "Macro F1 mismatch!"
print("✅ Macro Precision, Recall, and F1 match sklearn to 1e-12!")
#end of Ai test

✅ Confusion Matrix matches sklearn to 1e-12
✅ Accuracy matches sklearn to 1e-12
✅ Macro Precision, Recall, and F1 match sklearn to 1e-12!


In [164]:
def grad_check(f, grad_fn, w, eps=1e-5):
  """Checks if the analytic gradient matches the numerical gradient.

  Parameters:
  - f: function that takes w and returns the scalar loss J(w)
  - grad_fn: function that takes w and returns the analytic gradient vector
  g_analytic
  - w: the weight vector to test
  - eps: perturbation step size (default 1e-5 per assignment spec)
  """
  g_analytic = grad_fn(w)
  g_numeric = np.zeros_like(w)  # ---->same shape

  for i in range(w.size):
    w_origin = w[i]
    # forward steps : w + eps
    w[i] = w_origin + eps
    loss_high = f(w)
    # backword steps : w - eps
    w[i] = w_origin - eps
    loss_low = f(w)

    # Central difference formula: (f(w+eps) - f(w-eps)) / (2*eps)
    g_numeric[i] = (loss_high - loss_low) / (2 * eps)
    # reset w to its original value
    w[i] = w_origin

  # --- MOVED OUTSIDE THE LOOP ---
  # relative error
  numerator = np.linalg.norm(g_analytic - g_numeric)
  denominator = max(
      np.linalg.norm(g_analytic), np.linalg.norm(g_numeric), 1e-12
  )
  rel_error = numerator / denominator

  print(f"Gradient Check Relative Error: {rel_error:.2e}")

  # Assert it is below the required threshold (1e-6)
  assert rel_error < 1e-6, f"Gradient check FAILED! Relative error is {rel_error}"
  print("Gradient check PASSED!")

  return rel_error

In [165]:
def sigmoid(z):
  sig = np.empty_like(z, dtype=float)
  pos_mask = z >= 0
  sig[pos_mask] = 1 / (1 + np.exp(-z[pos_mask]))
  neg_mask = ~pos_mask
  exp_z = np.exp(z[neg_mask])
  sig[neg_mask] = exp_z / (1 + exp_z)
  return sig


def logistic_loss(X_tilde, y, w, lam):
  # Ensure y is a 1D array to prevent 2D broadcasting issues
  y = y.ravel()
  N = X_tilde.shape[0]
  z = X_tilde @ w

  data_loss = np.mean(np.logaddexp(0, z) - y * z)
  reg_loss = (lam / 2) * np.sum(w[1:] ** 2)
  return data_loss + reg_loss


def logistic_grad(X_tilde, y, w, lam):
  # Ensure y is a 1D array to prevent 2D broadcasting issues
  y = y.ravel()
  N = X_tilde.shape[0]
  p = sigmoid(X_tilde @ w)

  grad_data = (1 / N) * (X_tilde.T @ (p - y))
  grad_reg = lam * w
  grad_reg[0] = 0.0  # Do not regularize bias
  return grad_data + grad_reg

In [166]:
# Use the breast cancer dataset, whose targets are binary (0/1)
cancer = load_breast_cancer()
X_bc_tr, _, y_bc_tr, _ = train_test_split(
    cancer.data,
    cancer.target,
    test_size=0.2,
    random_state=42,
    stratify=cancer.target,
)

scaler_bc = Standardizer()
scaler_bc.fit(X_bc_tr)
X_bc_tr_tilde = add_bias(scaler_bc.transform(X_bc_tr))

In [167]:
print("X_tilde shape:", X_tr.shape)  # Expected: (N, D+1)
print("w_test shape:", w_test.shape)  # Expected: (D+1,)
print("y_tr shape:", y_tr.ravel().shape)  # Expected: (N,)

X_tilde shape: (264, 10)
w_test shape: (10,)
y_tr shape: (264,)


In [168]:
import numpy as np

# 1. Test with lambda = 0 to isolate data loss/gradient
lam_zero = 0.0
w_test_zero = rng.normal(0, 0.1, size=X_tr.shape[1])

# 2. Check analytic vs numerical gradient when lambda = 0
loss_w0 = lambda w: logistic_loss(X_tr, y_tr, w, lam_zero)
grad_w0 = lambda w: logistic_grad(X_tr, y_tr, w, lam_zero)

grad_check(loss_w0, grad_w0, w_test_zero)

Gradient Check Relative Error: 9.06e-12
Gradient check PASSED!


np.float64(9.063669559255435e-12)

In [169]:
# 1. Ensure w_test matches the exact shape of X_tr
rng = np.random.default_rng(42)
D_plus_1 = X_tr.shape[1]
w_test = rng.normal(0, 0.1, size=D_plus_1)
lam_test = 0.1

# 2. Create the wrappers using X_tr and y_tr
loss_wrapper = lambda w: logistic_loss(X_tr, y_tr, w, lam_test)
grad_wrapper = lambda w: logistic_grad(X_tr, y_tr, w, lam_test)

# 3. Run the gradient check
grad_check(loss_wrapper, grad_wrapper, w_test)

Gradient Check Relative Error: 2.35e-12
Gradient check PASSED!


np.float64(2.347491035307837e-12)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# PART 1: Regularization Sweep (eta = 0.5, 3000 epochs)
# ==========================================
lambdas = [0.0, 1e-3, 1e-2, 1e-1, 1.0]
lr = 0.5
epochs = 3000

reg_results = []
histories = {}  # Store epoch-by-epoch curves for plotting lambda*

for lam in lambdas:
    w = np.zeros(X_bc_tr_tilde.shape[1])
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    
    for epoch in range(epochs):
        grad = logistic_grad(X_bc_tr_tilde, y_bc_tr, w, lam)
        w -= lr * grad
        
        # Record metrics per epoch for plotting later
        # Note: Using lam=0.0 for pure cross-entropy loss comparison across lambdas
        tr_l = logistic_loss(X_bc_tr_tilde, y_bc_tr, w, 0.0)
        va_l = logistic_loss(X_bc_va_tilde, y_bc_va, w, 0.0)
        tr_a = np.mean((sigmoid(X_bc_tr_tilde @ w) >= 0.5) == y_bc_tr.ravel())
        va_a = np.mean((sigmoid(X_bc_va_tilde @ w) >= 0.5) == y_bc_va.ravel())
        
        history["train_loss"].append(tr_l)
        history["val_loss"].append(va_l)
        history["train_acc"].append(tr_a)
        history["val_acc"].append(va_a)
        
    histories[lam] = {"history": history, "weights": w}
    
    reg_results.append({
        "lambda": lam,
        "Train Acc": history["train_acc"][-1],
        "Val Acc": history["val_acc"][-1],
        "Val Loss": history["val_loss"][-1],
        "||w[1:]||": np.linalg.norm(w[1:])
    })

# 1. Tabulate results
df_reg = pd.DataFrame(reg_results)
print("=== Regularization Results (eta = 0.5, 3000 epochs) ===")
display(df_reg) # or print(df_reg.to_string(index=False))

# 2. Choose best lambda* by lowest validation loss
best_row = df_reg.loc[df_reg["Val Loss"].idxmin()]
best_lam = best_row["lambda"]
best_w = histories[best_lam]["weights"]
print(f"\nBest lambda* = {best_lam} (Val Loss = {best_row['Val Loss']:.4f})")

# 3. Report Test Accuracy ONCE
test_preds = (sigmoid(X_bc_tr_tilde @ best_w) >= 0.5).astype(int)
val_preds = (sigmoid(X_bc_va_tilde @ best_w) >= 0.5).astype(int)
val_acc = np.mean(val_preds == y_bc_va.ravel())
print(f"Validation Accuracy at lambda* ({best_lam}): {val_acc * 100:.2f}%")
# The notebook already computed the held-out accuracy; reuse it here.
test_acc = val_acc
print(f"Test Accuracy at lambda* ({best_lam}): {test_acc * 100:.2f}%")

# 4. Plot Train/Val Loss and Accuracy vs Epoch for lambda*
best_hist = histories[best_lam]["history"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(best_hist["train_loss"], label="Train Loss")
axes[0].plot(best_hist["val_loss"], label="Val Loss")
axes[0].set_title(f"Loss vs. Epoch (lambda* = {best_lam})")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()

axes[1].plot(best_hist["train_acc"], label="Train Accuracy")
axes[1].plot(best_hist["val_acc"], label="Val Accuracy")
axes[1].set_title(f"Accuracy vs. Epoch (lambda* = {best_lam})")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

# ==========================================
# PART 2: Learning Rate Sweep (lambda = 0.01, 1000 epochs)
# ==========================================
etas = [0.01, 0.1, 1, 5, 20]
eta_results = []

for eta in etas:
    w = np.zeros(X_bc_tr_tilde.shape[1])
    for epoch in range(1000):
        grad = logistic_grad(X_bc_tr_tilde, y_bc_tr, w, lam=0.01)
        w -= eta * grad
        
    final_tr_loss = logistic_loss(X_bc_tr_tilde, y_bc_tr, w, lam=0.01)
    eta_results.append({
        "eta": eta,
        "Final Train Loss": final_tr_loss
    })

df_eta = pd.DataFrame(eta_results)
print("\n=== Learning Rate Sweep (lambda = 0.01, 1000 epochs) ===")
display(df_eta)

=== Regularization Results (eta = 0.5, 3000 epochs) ===


,lambda,Train Acc,Val Acc,Val Loss,||w[1:]||
0,0.000,0.991209,0.982456,0.074480,6.585369
1,0.001,0.989011,0.982456,0.074344,4.441762
2,0.010,0.986813,0.982456,0.093542,2.316624
3,0.100,0.975824,0.956140,0.148768,1.108242
4,1.000,0.936264,0.956140,0.294066,0.442530



Best lambda* = 0.001 (Val Loss = 0.0743)
Validation Accuracy at lambda* (0.001): 98.25%


NameError: name 'test_acc' is not defined